In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet

pd.set_option('display.max_columns', None)

In [2]:
from google.colab import files
uploaded = files.upload()

# reading the csv
df = pd.read_csv('covid_19_data.csv')
df.head()

Saving covid_19_data.csv to covid_19_data.csv


,SNo,ObservationDate,Province/State,Country/Region,Last Update,Confirmed,Deaths,Recovered
0,1,01/22/2020,Anhui,Mainland China,1/22/2020 17:00,1.0,0.0,0.0
1,2,01/22/2020,Beijing,Mainland China,1/22/2020 17:00,14.0,0.0,0.0
2,3,01/22/2020,Chongqing,Mainland China,1/22/2020 17:00,6.0,0.0,0.0
3,4,01/22/2020,Fujian,Mainland China,1/22/2020 17:00,1.0,0.0,0.0
4,5,01/22/2020,Gansu,Mainland China,1/22/2020 17:00,0.0,0.0,0.0


In [3]:
print(df.shape)

(306429, 8)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306429 entries, 0 to 306428
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   SNo              306429 non-null  int64  
 1   ObservationDate  306429 non-null  object 
 2   Province/State   228326 non-null  object 
 3   Country/Region   306429 non-null  object 
 4   Last Update      306429 non-null  object 
 5   Confirmed        306429 non-null  float64
 6   Deaths           306429 non-null  float64
 7   Recovered        306429 non-null  float64
dtypes: float64(3), int64(1), object(4)
memory usage: 18.7+ MB


In [5]:
print(df.duplicated().sum())

0


In [6]:
print(df.isnull().sum())

SNo                    0
ObservationDate        0
Province/State     78103
Country/Region         0
Last Update            0
Confirmed              0
Deaths                 0
Recovered              0
dtype: int64


In [7]:
# fixing dates datatype
df['ObservationDate'] = pd.to_datetime(df['ObservationDate'])

In [8]:
# dropping columns i dont need
df = df.drop(['SNo', 'Last Update'], axis=1)

In [9]:
# rename columns
df = df.rename(columns={'ObservationDate': 'Date', 'Country/Region': 'Country', 'Province/State': 'State'})

In [10]:
# fill the blanks in province
df['State'] = df['State'].fillna('Unknown')

In [11]:
print(df['Date'].min(), df['Date'].max())
df.head()

2020-01-22 00:00:00 2021-05-29 00:00:00


,Date,State,Country,Confirmed,Deaths,Recovered
0,2020-01-22,Anhui,Mainland China,1.0,0.0,0.0
1,2020-01-22,Beijing,Mainland China,14.0,0.0,0.0
2,2020-01-22,Chongqing,Mainland China,6.0,0.0,0.0
3,2020-01-22,Fujian,Mainland China,1.0,0.0,0.0
4,2020-01-22,Gansu,Mainland China,0.0,0.0,0.0


In [12]:
# sum all country for each date
world = df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered']].sum().reset_index()

world.tail()

,Date,Confirmed,Deaths,Recovered
489,2021-05-25,167848207.0,3485788.0,104824935.0
490,2021-05-26,168416423.0,3498544.0,105380748.0
491,2021-05-27,168970791.0,3511297.0,106078106.0
492,2021-05-28,169470725.0,3523117.0,106633069.0
493,2021-05-29,169951560.0,3533619.0,107140669.0


In [13]:
# line chart for world trend

fig = px.line(world, x='Date', y=['Confirmed', 'Deaths', 'Recovered'],
              title='World COVID-19 Trend')
fig.show()

In [14]:
# daily change in recovered
world['Rec_change'] = world['Recovered'].diff()

# the day with the biggest drop
worst = world.loc[world['Rec_change'].idxmin()]
print(worst['Date'], worst['Rec_change'])

2020-12-15 00:00:00 -6041367.0


In [15]:
d = worst['Date']

# recovered per country, day before and that day
before = df[df['Date'] == d - pd.Timedelta(days=1)].groupby('Country')['Recovered'].sum()
on_day = df[df['Date'] == d].groupby('Country')['Recovered'].sum()

# biggest negative changes
print((on_day - before).sort_values().head(5))

Country
US                    -6399531.0
Andorra                      0.0
Australia                    0.0
Antigua and Barbuda          0.0
Brunei                       0.0
Name: Recovered, dtype: float64


In [16]:
# only india rows
india_df = df[df['Country'] == 'India']
india = india_df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered']].sum().reset_index()

india.tail()

,Date,Confirmed,Deaths,Recovered
481,2021-05-25,27157795.0,311388.0,24350816.0
482,2021-05-26,27369093.0,315235.0,24633951.0
483,2021-05-27,27555457.0,318895.0,24893410.0
484,2021-05-28,27729247.0,322512.0,25178011.0
485,2021-05-29,27894800.0,325972.0,25454320.0


In [17]:
fig = px.line(india, x='Date', y=['Confirmed', 'Deaths', 'Recovered'],
              title='India COVID-19 Trend')
fig.show()

In [18]:
#  world rates
world['Recovery_Rate'] = world['Recovered'] / world['Confirmed'] * 100
world['Death_Rate'] = world['Deaths'] / world['Confirmed'] * 100

#India rates
india['Recovery_Rate'] = india['Recovered'] / india['Confirmed'] * 100
india['Death_Rate'] = india['Deaths'] / india['Confirmed'] * 100

# world without US
no_us = df[df['Country'] != 'US'].groupby('Date')[['Confirmed', 'Deaths', 'Recovered']].sum().reset_index()
no_us['Recovery_Rate'] = no_us['Recovered'] / no_us['Confirmed'] * 100

In [19]:
#Visualizing
fig = go.Figure()
fig.add_trace(go.Scatter(x=world['Date'], y=world['Recovery_Rate'], name='World'))
fig.add_trace(go.Scatter(x=no_us['Date'], y=no_us['Recovery_Rate'], name='World (no US)'))
fig.add_trace(go.Scatter(x=india['Date'], y=india['Recovery_Rate'], name='India'))
fig.update_layout(title='Recovery Rate (%)')
fig.show()

In [20]:
# new cases per day
world['New_Cases'] = world['Confirmed'].diff()
india['New_Cases'] = india['Confirmed'].diff()

#  average a week to smooth the noise
world['New_Avg'] = world['New_Cases'].rolling(7).mean()
india['New_Avg'] = india['New_Cases'].rolling(7).mean()

print(india['New_Cases'].min())

0.0


In [21]:
fig = go.Figure()
fig.add_trace(go.Bar(x=india['Date'], y=india['New_Cases'], name='Daily'))
fig.add_trace(go.Scatter(x=india['Date'], y=india['New_Avg'], name='7-day avg'))
fig.update_layout(title='India Daily New Cases')
fig.show()

In [22]:
# biggest single day in india
peak_day = india.loc[india['New_Cases'].idxmax()]
print(peak_day['Date'], peak_day['New_Cases'])

2021-05-06 00:00:00 414188.0


In [23]:
# latest date only
last = df[df['Date'] == df['Date'].max()]

# total confirmed per country top 10
top10 = last.groupby('Country')['Confirmed'].sum().nlargest(10).reset_index()

fig = px.bar(top10, x='Country', y='Confirmed', title='Top 10 Countries by Confirmed Cases')
fig.show()
print(top10)

     Country   Confirmed
0         US  33251939.0
1      India  27894800.0
2     Brazil  16471600.0
3     France   5719877.0
4     Turkey   5235978.0
5     Russia   4995613.0
6         UK   4496823.0
7      Italy   4213055.0
8  Argentina   3732263.0
9    Germany   3684672.0


In [24]:
# keep only needed columns from each table
w = world[['Date', 'Confirmed', 'New_Cases']]
i = india[['Date', 'Confirmed', 'New_Cases']]

# join on date
merged = pd.merge(w, i, on='Date', suffixes=('_World', '_India'))

# india's share of world cases in %
merged['India_Share'] = merged['Confirmed_India'] / merged['Confirmed_World'] * 100

merged.tail()

,Date,Confirmed_World,New_Cases_World,Confirmed_India,New_Cases_India,India_Share
481,2021-05-25,167848207.0,531845.0,27157795.0,208921.0,16.179973
482,2021-05-26,168416423.0,568216.0,27369093.0,211298.0,16.250846
483,2021-05-27,168970791.0,554368.0,27555457.0,186364.0,16.307823
484,2021-05-28,169470725.0,499934.0,27729247.0,173790.0,16.362264
485,2021-05-29,169951560.0,480835.0,27894800.0,165553.0,16.413383


In [25]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=world['Date'], y=world['Death_Rate'], name='World'))
fig.add_trace(go.Scatter(x=india['Date'], y=india['Death_Rate'], name='India'))
fig.update_layout(title='Death Ratein  %')
fig.show()

In [26]:
#Preparing dataset for prediction

prophet_df = india[['Date', 'Confirmed']].rename(columns={'Date': 'ds', 'Confirmed': 'y'})

# test on last 7 days
train = prophet_df.iloc[:-7]
test = prophet_df.iloc[-7:]

print(train.tail(1))
print(test)

            ds           y
478 2021-05-22  26530132.0
            ds           y
479 2021-05-23  26752447.0
480 2021-05-24  26948874.0
481 2021-05-25  27157795.0
482 2021-05-26  27369093.0
483 2021-05-27  27555457.0
484 2021-05-28  27729247.0
485 2021-05-29  27894800.0


In [27]:
# yearly off, data is only ~16 months
m = Prophet(yearly_seasonality=False, daily_seasonality=False)
m.fit(train)

# 7 days ahead of the training data
future = m.make_future_dataframe(periods=7)
forecast = m.predict(future)

forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7)

,ds,yhat,yhat_lower,yhat_upper
479,2021-05-23,2.171771e+07,2.037196e+07,2.315139e+07
480,2021-05-24,2.184654e+07,2.042390e+07,2.322331e+07
481,2021-05-25,2.198048e+07,2.062825e+07,2.337880e+07
482,2021-05-26,2.211895e+07,2.064445e+07,2.346337e+07
483,2021-05-27,2.225596e+07,2.077044e+07,2.369333e+07
484,2021-05-28,2.239299e+07,2.097311e+07,2.374803e+07
485,2021-05-29,2.253072e+07,2.107347e+07,2.392813e+07


In [28]:
pred = forecast['yhat'].iloc[-7:].values
actual = test['y'].values

mae = np.mean(np.abs(actual - pred))
rmse = np.sqrt(np.mean((actual - pred) ** 2))
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print('MAE:', mae)
print('RMSE:', rmse)
print('MAPE %:', mape)

MAE: 5223482.090014473
RMSE: 5224731.892799054
MAPE %: 19.100808802722177


In [29]:
## we got a bad model so new model with changepoints

In [30]:
# changepoints allowed in 95% of data, more flexible trend
m2 = Prophet(yearly_seasonality=False, daily_seasonality=False,
             changepoint_range=0.95, changepoint_prior_scale=0.5)
m2.fit(train)

future2 = m2.make_future_dataframe(periods=7)
forecast2 = m2.predict(future2)

pred2 = forecast2['yhat'].iloc[-7:].values

mae2 = np.mean(np.abs(actual - pred2))
rmse2 = np.sqrt(np.mean((actual - pred2) ** 2))
mape2 = np.mean(np.abs((actual - pred2) / actual)) * 100

print('MAE:', mae2)
print('RMSE:', rmse2)
print('MAPE %:', mape2)

MAE: 1171514.2554560027
RMSE: 1219995.9437708138
MAPE %: 4.2677524720156965


In [31]:
# refit on full data with the changed settings
final = Prophet(yearly_seasonality=False, daily_seasonality=False,
                changepoint_range=0.95, changepoint_prior_scale=0.5)
final.fit(prophet_df)

future_f = final.make_future_dataframe(periods=7)
forecast_f = final.predict(future_f)

forecast_f[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7)

,ds,yhat,yhat_lower,yhat_upper
486,2021-05-30,2.885097e+07,2.873374e+07,2.896207e+07
487,2021-05-31,2.914596e+07,2.903127e+07,2.925842e+07
488,2021-06-01,2.944770e+07,2.933395e+07,2.954602e+07
489,2021-06-02,2.975462e+07,2.964026e+07,2.987314e+07
490,2021-06-03,3.006029e+07,2.994058e+07,3.016904e+07
491,2021-06-04,3.036646e+07,3.025008e+07,3.048933e+07
492,2021-06-05,3.066984e+07,3.054498e+07,3.080108e+07


In [32]:
#forecast chart
fig = go.Figure()
fig.add_trace(go.Scatter(x=prophet_df['ds'], y=prophet_df['y'], name='Actual'))
fig.add_trace(go.Scatter(x=forecast_f['ds'].iloc[-7:], y=forecast_f['yhat'].iloc[-7:], name='Forecast'))
fig.add_trace(go.Scatter(x=forecast_f['ds'].iloc[-7:], y=forecast_f['yhat_upper'].iloc[-7:], name='Upper', line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=forecast_f['ds'].iloc[-7:], y=forecast_f['yhat_lower'].iloc[-7:], name='Lower', line=dict(dash='dot')))
fig.update_layout(title='India Confirmed Cases: 7-Day Forecast')
fig.show()

In [33]:
# model's fitted values on the data it trained on
check = forecast_f[['ds', 'yhat']].iloc[-12:-7].copy()

# real values for the same dates
check['actual'] = prophet_df['y'].iloc[-5:].values
check['gap'] = check['yhat'] - check['actual']

check

,ds,yhat,actual,gap
481,2021-05-25,2.731682e+07,27157795.0,159026.798364
482,2021-05-26,2.762374e+07,27369093.0,254646.500990
483,2021-05-27,2.792941e+07,27555457.0,373955.143560
484,2021-05-28,2.823558e+07,27729247.0,506333.736206
485,2021-05-29,2.853896e+07,27894800.0,644163.564544
